# 02 — Interpolation impaired / vulnerability profile (E2)
Results §2, Fig 2B–C, Fig 3. **RECOMPUTE** (canonical FE-6 uniform + OLS).

**Provenance.** Built by `~/.claude/skills/repro-notebook/SKILL.md` (Phase 3).
Companion docs: `docs/PAPER/repro/{MANIFEST.md, MAP.md, REPORT.md}`.
Target paper: `docs/PAPER/main.tex` → `Results/results_v4.tex`.
Helpers: `docs/PAPER/repro/_repro_util.py`.

**Modes.** `RECOMPUTE` = computed locally from C010 amplitudes. `LOAD+VERIFY` =
read the committed result JSON and compare to the printed paper value. SRM/BrainIAK
numbers are read from committed JSON (no MPI in this kernel).

**Source & code map**

| id | reported (results_v4.tex L38–40) | source | mode |
|---|---|---|---|
| E2.1 | HC adj 0.47±0.05; hV4 above-chance **p=0.008** (1,000 per-subj perms) | C010 amplitudes; `perm_definitive_hv4_null.npy` | RECOMPUTE / LOAD |
| E2.2 | V1 **p=0.164** (n.s.); V2,V3 below chance | C010 amplitudes; `perm_v1_null.npy` | RECOMPUTE / LOAD |
| E2.3/2.4 | deutan 0.25 (CH −1.84/0.063/−1.99); protan 0.13 (−2.91/0.017/−3.14) | C010 amplitudes | RECOMPUTE |
| E2.5/2.6 | blue/purple/magenta=0; per-hue CH none sig | C010 amplitudes | RECOMPUTE |

Reconciliation note (2026-06-28): an earlier draft reported hV4 p=0.044 (voxel_corr
8! perm) and CH −1.58/0.082 & −2.48/0.024. The manuscript has since adopted the
adjacent-accuracy values reproduced here; `reported=` targets track the current tex.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))   # docs/PAPER/repro
import numpy as np
import _repro_util as U
U._RESULTS.clear()   # fresh check log per notebook
print("repo:", U.REPO)

repo: /Users/jinilkim/Library/CloudStorage/OneDrive-Personal/Projects/colorBlind_analysis


### E2.1 / E2.2 — hV4 HC adjacent accuracy 0.47±0.05 (n=6); above-chance p=0.008
Canonical above-chance test = **adjacent-accuracy** under 1,000 per-subject label
permutations (FE-6 uniform + OLS), matching `results_v4.tex` L38 and `PERMUTATIONS.md`.
Committed null draws: `perm_definitive_hv4_null.npy` (hV4), `perm_v1_null.npy` (V1).
p = add-one `(#{null≥obs}+1)/(N+1)` (Phipson & Smyth 2010).

In [2]:
hc = U.hc_adjacent_matrix()           # (6,8) hV4, sub-07 excluded
hc_overall = hc.mean(axis=1)
U.check("E2.1 HC adj mean", hc_overall.mean(), 0.47, tol=0.02)
print("HC SEM=%.3f (paper 0.05)" % (hc_overall.std(ddof=1)/np.sqrt(len(hc))))

# hV4 adjacent-accuracy above-chance: committed 1,000 per-subject label perms -> p=0.008
obs_hv4 = hc_overall.mean()
null_hv4 = np.load(U.REPRO / "perm_definitive_hv4_null.npy")
p_hv4 = (np.sum(null_hv4 >= obs_hv4) + 1) / (len(null_hv4) + 1)
U.check("E2.1 hV4 adj above-chance p", p_hv4, 0.008, tol=0.003)
print("  obs=%.4f null_mean=%.4f N=%d (per-subject label perm, FE-6 OLS)" % (obs_hv4, null_hv4.mean(), len(null_hv4)))

# E2.2 V1 not above chance (n=7, sub-07 included); V2,V3 below chance by inspection
v1 = np.array([U.adjacent_accuracy_profile(s, roi="V1") for s in U.HC])
obs_v1 = v1.mean(axis=1).mean()
null_v1 = np.load(U.REPRO / "perm_v1_null.npy")
p_v1 = (np.sum(null_v1 >= obs_v1) + 1) / (len(null_v1) + 1)
U.check("E2.2 V1 adj above-chance p (n.s.)", p_v1, 0.164, tol=0.01)
print("  V1 obs=%.4f p=%.3f -> not above chance (V2,V3 < 0.375 chance by inspection)" % (obs_v1, p_v1))

[OK ] E2.1 HC adj mean: produced=0.46527777777777773  reported=0.47
HC SEM=0.044 (paper 0.05)
[OK ] E2.1 hV4 adj above-chance p: produced=0.007992007992007992  reported=0.008
  obs=0.4653 null_mean=0.3473 N=1000 (per-subject label perm, FE-6 OLS)
[OK ] E2.2 V1 adj above-chance p (n.s.): produced=0.16383616383616384  reported=0.164
  V1 obs=0.3929 p=0.164 -> not above chance (V2,V3 < 0.375 chance by inspection)


### E2.3 / E2.4 — deutan 0.25 (CH −1.84/0.063/−1.99), protan 0.13 (−2.91/0.017/−3.14)
Overall Crawford–Howell single-case vs n=6 HC. Targets are the current `results_v4.tex`
L40 values (the manuscript adopted these reproduced statistics).

In [3]:
d08 = U.adjacent_accuracy_profile("08"); d09 = U.adjacent_accuracy_profile("09")
U.check("E2.3 deutan overall", d08.mean(), 0.25, tol=0.01)
U.check("E2.4 protan overall", d09.mean(), 0.13, tol=0.01)
CH_TARGET = {"deutan": (-1.84, 0.063, -1.99), "protan": (-2.91, 0.017, -3.14)}
for name, v in [("deutan", d08), ("protan", d09)]:
    t, p, dcc = U.crawford_howell(v.mean(), hc.mean(axis=1))
    rt, rp, rd = CH_TARGET[name]
    U.check(f"E2.3/2.4 {name} CH t", t, rt, tol=0.02)
    U.check(f"E2.3/2.4 {name} CH p", p, rp, tol=0.003)
    U.check(f"E2.3/2.4 {name} CH d_cc", dcc, rd, tol=0.02)
    print(f"  {name} overall CH: t={t:+.2f} p={p:.3f} d_cc={dcc:+.2f}")

[OK ] E2.3 deutan overall: produced=0.25  reported=0.25
[OK ] E2.4 protan overall: produced=0.125  reported=0.13
[OK ] E2.3/2.4 deutan CH t: produced=-1.838864196761385  reported=-1.84
[OK ] E2.3/2.4 deutan CH p: produced=0.06266526865048642  reported=0.063
[OK ] E2.3/2.4 deutan CH d_cc: produced=-1.9862003397994314  reported=-1.99
  deutan overall CH: t=-1.84 p=0.063 d_cc=-1.99
[OK ] E2.3/2.4 protan CH t: produced=-2.906591794880899  reported=-2.91
[OK ] E2.3/2.4 protan CH p: produced=0.01676619880185896  reported=0.017
[OK ] E2.3/2.4 protan CH d_cc: produced=-3.139477956457166  reported=-3.14
  protan overall CH: t=-2.91 p=0.017 d_cc=-3.14


### E2.5 / E2.6 — per-hue: blue/purple/magenta zero; no individual hue significant

In [4]:
for name, v in [("deutan", d08), ("protan", d09)]:
    print(f"-- {name} --")
    for c in (U.HUE_NAMES.index(h) for h in ["blue", "purple", "magenta"]):
        t, p, dcc = U.crawford_howell(v[c], hc[:, c])
        U.check(f"E2.5 {name} {U.HUE_NAMES[c]} acc=0", v[c], 0.0, tol=1e-9)
        print(f"     {U.HUE_NAMES[c]:7s} CH: t={t:+.2f} p={p:.3f} d_cc={dcc:+.2f}  sig={'YES' if p<0.05 else 'no'}")
print("E2.6 target: NO individual hue reaches p<0.05  (blue p=0.072, purple 0.205, magenta 0.122)")

-- deutan --
[OK ] E2.5 deutan blue acc=0: produced=0.0  reported=0.0
     blue    CH: t=-1.73 p=0.072 d_cc=-1.87  sig=no
[OK ] E2.5 deutan purple acc=0: produced=0.0  reported=0.0
     purple  CH: t=-0.90 p=0.205 d_cc=-0.97  sig=no
[OK ] E2.5 deutan magenta acc=0: produced=0.0  reported=0.0
     magenta CH: t=-1.32 p=0.122 d_cc=-1.43  sig=no
-- protan --
[OK ] E2.5 protan blue acc=0: produced=0.0  reported=0.0
     blue    CH: t=-1.73 p=0.072 d_cc=-1.87  sig=no
[OK ] E2.5 protan purple acc=0: produced=0.0  reported=0.0
     purple  CH: t=-0.90 p=0.205 d_cc=-0.97  sig=no
[OK ] E2.5 protan magenta acc=0: produced=0.0  reported=0.0
     magenta CH: t=-1.32 p=0.122 d_cc=-1.43  sig=no
E2.6 target: NO individual hue reaches p<0.05  (blue p=0.072, purple 0.205, magenta 0.122)


In [5]:
U.summary()


=== 17/17 checks reproduced ===
